# Sliding Window Statistics

Imports

In [78]:
!uv pip install ipywidgets nbformat
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt 
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go 
%matplotlib inline

Using Python 3.11.13 environment at: /Users/dballen/.pyenv/versions/3.11.13/envs/env
Audited 2 packages in 47ms


# Loading the Data
First we will load the data of 100 runs for each sliding window calculation

In [79]:
df = pd.read_csv("../../data/rmse_trials_100runs.csv")

In [80]:
df = df.drop(columns={'Unnamed: 0'}, axis=1)
df

,feature,window_size,rmse,run
0,FIRE_Acres_Burned,1,2.640562,0
1,FIRE_Acres_Burned,1,2.656766,1
2,FIRE_Acres_Burned,1,2.567147,2
3,FIRE_Acres_Burned,1,2.702376,3
4,FIRE_Acres_Burned,1,2.651300,4
...,...,...,...,...
11995,POUNDS_PRODUCT_APPLIED,12,2.037867,95
11996,POUNDS_PRODUCT_APPLIED,12,2.673531,96
11997,POUNDS_PRODUCT_APPLIED,12,2.223721,97
11998,POUNDS_PRODUCT_APPLIED,12,2.757697,98


In [81]:
feature_name = "POUNDS_PRODUCT_APPLIED"
window_size_val = 8

hist_data = df.loc[(df['feature'] == feature_name) & (df['window_size'] == window_size_val)]
fig = px.histogram(hist_data, x = 'rmse', title=f"Histogram of RMSE for {feature_name} || window size: {window_size_val}")
fig.show()

In [82]:
feature_names = set(df['feature'])
n_feat = len(feature_names)
window_sizes = set(df['window_size'])
n_windows = len(window_sizes)


We now want to make a subplot of n_feat rows x n_window columns

In [83]:
titles = []
for feat in feature_names:
    for ws in window_sizes:
        titles.append(f"feat: {feat} | ws: {ws}")

fig = make_subplots(rows = n_feat, cols = n_windows, subplot_titles=titles)
for idx, feat in enumerate(feature_names):
  for ws in window_sizes:
    hist_data = df.loc[(df['feature'] == feat) & (df['window_size'] == ws)]
    fig.add_trace(
      go.Histogram(x = hist_data['rmse']),
      row = idx+1, col = ws
    )

fig.update_layout(height = 1200, width = 1400, title_text = "Feature vs Window_Size Histograms")
fig.show()

In [84]:
# 1. Increase vertical_spacing to prevent title overlap
# 2. Use shared_xaxes and shared_yaxes to clean up the interior of the grid
titles = []
for feat in feature_names:
    for ws in window_sizes:
        # Using <br> for a line break between the feature and window size
        titles.append(f"<b>Feat:</b> {feat}<br><b>WS:</b> {ws}")

fig = make_subplots(
    rows=n_feat, 
    cols=n_windows, 
    subplot_titles=titles,
    vertical_spacing=0.08,  # Increase this further to accommodate the two-line titles
    shared_xaxes=True,
    shared_yaxes=True
)

for idx, feat in enumerate(feature_names):
    # Use enumerate for the column to ensure you use 1, 2, 3... instead of the ws value
    for col_idx, ws in enumerate(window_sizes):
        hist_data = df.loc[(df['feature'] == feat) & (df['window_size'] == ws)]
        
        fig.add_trace(
            go.Histogram(
                x=hist_data['rmse'],
                marker_color='#636EFA', # Consistent branding color
                opacity=0.75,
                nbinsx=20 # Controlling bins helps prevent 'choppy' histograms
            ),
            row=idx+1, col=col_idx+1
        )

# Use a clean template and adjust margins to utilize the full width
fig.update_layout(
    template="plotly_white",
    height=1200, 
    width=1400, 
    title_text="Model Error Distribution: Feature vs. Window Size",
    title_x=0.5, # Center the main title
    showlegend=False,
    margin=dict(l=50, r=50, t=100, b=50)
)

# Optional: Adjust font size of subplot titles if they are still too big
fig.update_annotations(font_size=8)

fig.show()

In [88]:
# 1. Group by feature and window size, then calculate the mean RMSE
hierarchical_df = df.groupby(['feature', 'window_size'])['rmse'].mean().to_frame()

# Optional: Add standard deviation to see the 'stability' alongside the mean
hierarchical_df['rmse_std'] = df.groupby(['feature', 'window_size'])['rmse'].std()

# Rename the mean column for clarity
hierarchical_df.rename(columns={'rmse': 'avg_rmse'}, inplace=True)

hierarchical_df.head()
    

avg_rmse  rmse_std
feature  window_size                    
AQI_PM10 1            2.753293  0.030118
         2            2.772549  0.038471
         3            3.033498  0.148134
         4            3.114847  0.362761
         5            2.882634  0.458601

In [90]:
# 1. Aggregate the data to get the average RMSE per combination
# We reset_index to keep 'feature' and 'window_size' as columns for plotting
agg_df = df.groupby(['feature', 'window_size'])['rmse'].mean().reset_index()

# 2. Create a faceted line plot
fig = px.line(
    agg_df, 
    x='window_size', 
    y='rmse', 
    color='feature', 
    facet_col='feature', 
    facet_col_wrap=4,      # Adjust this number based on how many features you have
    markers=True,          # Essential for spotting the specific window size points
    template="plotly_white",
    title="Average RMSE vs. Window Size"
    
)

# 3. Clean up the presentation
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1])) # Removes "feature=" from titles
fig.update_yaxes(matches=None) # Allows each feature to have its own scale (important if RMSE ranges vary)
fig.update_xaxes(type='category') # Keeps window sizes spaced evenly if they are non-linear (e.g., 12, 24, 48)

fig.update_layout(
    height=300 * (len(feature_names) // 4 + 1), # Dynamic height based on number of rows
    width=1200,
    showlegend=False,
    margin=dict(t=100)
)

fig.show()